# Intro 3：TabularX 回归任务 —— Diabetes 数据集

本教程演示 TabularX 在回归任务上的使用。Diabetes 数据集包含 442 名糖尿病患者的 10 个特征，目标是预测一年后的疾病进展。同时演示公平性评估与 SHAP 可解释性。


In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.datasets import load_diabetes
import tabularx as tbx
print("TabularX 已就绪")


TabularX 已就绪


In [2]:
data = load_diabetes(as_frame=True)
X, y = data.data, data.target
print(f"特征维度: {X.shape}")
print(f"样本数: {len(X)}")
print(f"特征数: {len(X.columns)}")
print(f"目标范围: {y.min():.1f} ~ {y.max():.1f}（均值: {y.mean():.1f}）")


特征维度: (442, 10)
样本数: 442
特征数: 10
目标范围: 25.0 ~ 346.0（均值: 152.1）


## 第一步：创建 AutoEstimator

回归任务传 `task="regression"`。主指标默认 RMSE（越小越好）。


In [3]:
ae = tbx.AutoEstimator(task="regression", output_dir="/mnt/c/Users/<user>/seadrive_root/<user>/我的资料库/课程资料/机器学习/大作业/introduction/intro3/output", pdf_report=True)
ae.param.model_search.set_param("n_trials", 30)
ae.param.model_search.set_param("top_k_models", 2)
ae.param.shap.set_param("top_k_features", 10)
ae.param.split.set_param("test_size", 0.2)
ae.param.split.set_param("validation_size", 0.25)
print(ae.param.overview())


  TabularX 参数总览

【Profile 阶段】
           text_unique_threshold =   200   #文本列判定：唯一值超过此数量 → 可能为文本列
               text_unique_ratio =   0.5   #文本列判定：唯一值占比（唯一值数/行数）超过此值 → 文本列
       int_categorical_threshold =    20   #整数列判定：唯一值数量 ≤ 此值 → 类别列
            scale_mean_threshold =   0.5   #Scale 判据：|mean| > 此值 → 需要标准化
             scale_std_threshold =   1.5   #Scale 判据：std > 此值 → 需要标准化
      high_cardinality_threshold =    50   #高基数判定：唯一值 > 此值 → high_cardinality
                  skew_threshold =   1.0   #偏斜判定：|skew| > 此值 → skewed
                 imbalance_ratio =   0.1   #类别不平衡：最小类别占比 < 此值 → imbalanced
          min_samples_regression =    10   #回归任务最少样本数（低于此值报错）
           min_samples_per_class =     2   #分类任务每类最少样本数（低于此值报错）

【Split 阶段】
                       test_size =   0.2   #测试集占比（占总样本的比例）
                 validation_size =  0.25   #验证集占比（相对于去掉测试集后剩余部分的比例）
                    random_state =    42   #随机种子（保证划分可复现）
               min_samples_split =     5   #训练/验证/测试集每集最少样本数

【Preproce

## 第二步：一键运行

回归流水线不会生成混淆矩阵和 ROC 曲线（仅分类有），但会生成 SHAP 解释图和调参轨迹。


In [4]:
ae.fit(X, y, sensitive_features=["age", "sex"])
print("运行完成！")


[I 2026-09-12 23:02:58,134] A new study created in memory with name: random_forest_regression
[I 2026-09-12 23:02:58,135] A new study created in memory with name: catboost_regression
[I 2026-09-12 23:02:58,137] A new study created in memory with name: lightgbm_regression
[I 2026-09-12 23:02:58,135] A new study created in memory with name: xgboost_regression


运行完成！


## 第三步：查看搜索结果

回归的排序指标是 RMSE（越小越好）。


In [5]:
result = ae.result
print(f"任务类型: {result.searched.task_type}")
print(f"主指标: {result.searched.primary_metric}（越小越好）\n")
print("=" * 70)
print(f"{"排名":>4} {"模型":>20} {"验证集RMSE":>12} {"测试集RMSE":>12}")
print("=" * 70)
for i, r in enumerate(result.searched.ranking, 1):
    v = r.validation_metrics.get("rmse", float("nan"))
    t = (r.test_metrics or {}).get("rmse", float("nan"))
    print(f"{i:>4} {r.model_name:>20} {v:>12.4f} {t:>12.4f}")
print(f"\n最佳模型: {result.searched.best_model_name}")


任务类型: regression
主指标: rmse（越小越好）

  排名                   模型      验证集RMSE      测试集RMSE
   1             catboost      52.0207      50.5234
   2             lightgbm      53.8050      52.4643
   3        random_forest      52.1630      53.1005
   4              xgboost      52.9729      53.2281

最佳模型: catboost


## 第四步：SHAP 可解释性

回归的 SHAP 分析展示各特征对预测值的影响。正值推高预测值，负值拉低预测值。


In [6]:
shap = result.shap
if shap:
    print(f"模型: {shap.model_name}")
    print(f"解释样本数: {shap.n_explain_samples}")
    print(f"Top-10 特征解释力占比: {shap.top_k_explained_ratio:.1%}\n")
    print(f"{"排名":>4} {"特征名":>25} {"平均 |SHAP|":>12}")
    print("-" * 45)
    for i, feat in enumerate(shap.top_k_features[:10], 1):
        val = shap.feature_importance.get(feat, 0.0)
        print(f"{i:>4} {feat:>25} {val:>12.4f}")
else:
    print("SHAP 未执行")


模型: catboost
解释样本数: 89
Top-10 特征解释力占比: 100.0%

  排名                       特征名    平均 |SHAP|
---------------------------------------------
   1                       bmi      19.0290
   2                        s5      18.3711
   3                        bp       9.9779
   4                       sex       9.3234
   5                        s3       8.2754
   6                        s4       6.4130
   7                       age       4.4791
   8                        s6       3.5106
   9                        s2       2.3952
  10                        s1       1.6556


## 第五步：公平性评估

对 age 和 sex 两个敏感特征进行公平性审计。


In [7]:
fairness = result.fairness
if fairness and fairness.enabled:
    for fr in fairness.feature_results:
        print(f"敏感特征: {fr.feature_name}")
        if fr.is_dropped:
            print(f"  丢弃原因: {fr.dropped_reason}")
            continue
        print(f"  DP差异: {fr.demographic_parity_diff:.4f} {"告警" if fr.dp_violation else "正常"}")
        for g in fr.groups:
            print(f"  {g.group_name}: n={g.n_samples} acc={g.accuracy:.4f}")
else:
    print("未启用公平性评估")


敏感特征: age
  DP差异: 148.0678 告警
  0.038075906433423026: n=2 acc=0.0000
  -0.07090024709715959: n=1 acc=0.0000
  0.027178291080364757: n=3 acc=0.0000
  -0.052737554842062495: n=2 acc=0.0000
  -0.027309785684926546: n=5 acc=0.0000
  -0.08543040090123728: n=1 acc=0.0000
  -0.06363517019512076: n=1 acc=0.0000
  -0.06726770864614018: n=1 acc=0.0000
  -0.06000263174410134: n=5 acc=0.0000
  0.03081082953138418: n=2 acc=0.0000
  -0.0018820165277906047: n=4 acc=0.0000
  -0.056370093293081916: n=2 acc=0.0000
  -0.04183993948900423: n=4 acc=0.0000
  0.06350367559055897: n=2 acc=0.0000
  0.012648137276287077: n=3 acc=0.0000
  -0.009147093429829445: n=2 acc=0.0000
  -0.030942324135945967: n=1 acc=0.0000
  -0.09632801625429555: n=2 acc=0.0000
  0.005383060374248237: n=2 acc=0.0000
  -0.10359309315633439: n=1 acc=0.0000
  -0.016412170331868287: n=1 acc=0.0000
  -0.04910501639104307: n=1 acc=0.0000
  0.016280675727306498: n=2 acc=0.0000
  0.01991321417832592: n=3 acc=0.0000
  0.04897352178648128: n=2 ac

## 第六步：报告文件


In [8]:
from pathlib import Path
print("生成的文件:")
for name, p in result.report_paths.items():
    ok = "OK" if Path(p).exists() else "MISS"
    print(f"  [{ok}] {name}: {p}")


生成的文件:
  [OK] profile: /mnt/c/Users/<user>/seadrive_root/<user>/我的资料库/课程资料/机器学习/大作业/introduction/intro3/output/reports/profile.html
  [OK] split: /mnt/c/Users/<user>/seadrive_root/<user>/我的资料库/课程资料/机器学习/大作业/introduction/intro3/output/reports/split.html
  [OK] preprocess: /mnt/c/Users/<user>/seadrive_root/<user>/我的资料库/课程资料/机器学习/大作业/introduction/intro3/output/reports/preprocess.html
  [OK] model_search: /mnt/c/Users/<user>/seadrive_root/<user>/我的资料库/课程资料/机器学习/大作业/introduction/intro3/output/reports/model_search.html
  [OK] explanation: /mnt/c/Users/<user>/seadrive_root/<user>/我的资料库/课程资料/机器学习/大作业/introduction/intro3/output/reports/explanation.html
  [OK] full: /mnt/c/Users/<user>/seadrive_root/<user>/我的资料库/课程资料/机器学习/大作业/introduction/intro3/output/report.html


## 总结

回归任务与分类任务使用完全一致的 API。区别在于：
- task 参数设为 `"regression"`
- 主指标默认 RMSE（越小越好）
- 不生成混淆矩阵和 ROC 曲线
- SHAP 分析展示对连续预测值的贡献方向
- 公平性评估同样支持回归模型
